# Data Cleaning
Recode the raw BRFSS 2023 values, create the diabetes target and save a clean CSV.

In [1]:
%pip install pandas numpy

  Using cached pandas-3.1.0-cp312-cp312-win_amd64.whl.metadata (18 kB)
  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached pandas-3.1.0-cp312-cp312-win_amd64.whl (10.7 MB)
Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl (12.6 MB)

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---

In [2]:
import os

import numpy as np
import pandas as pd

In [3]:
raw_data_folder = "../../data/raw"
processed_data_folder = "../../data/processed"
os.makedirs(processed_data_folder, exist_ok=True)

diabetes_data = pd.read_csv(raw_data_folder + "/Diabetes Health Indicators.csv")
rows_before = len(diabetes_data)
print(diabetes_data.shape)

(433323, 20)


In [4]:
diabetes_data.isna().sum()

DIABETE4         5
DIABTYPE    409048
_RFHYPE6         0
_RFCHOL3     51811
_CHOLCH3         0
_BMI5        40535
SMOKE100     19674
CVDSTRK3         4
_MICHD        4585
EXERANY2         2
PRIMINS1         5
MEDCOST1         2
GENHLTH          4
MENTHLTH         3
PHYSHLTH         3
DIFFWALK     16146
_SEX             0
_AGEG5YR         0
_EDUCAG          0
_INCOMG1         0
dtype: int64

In [5]:
# DIABTYPE is only asked to people who already have diabetes, so it leaks the target
diabetes_data = diabetes_data.drop(columns=["DIABTYPE"])

## Target

In [6]:
diabetes_data["DIABETE4"].value_counts(dropna=False).sort_index()

DIABETE4
1.0     59786
2.0      3253
3.0    358706
4.0     10594
7.0       683
9.0       296
NaN         5
Name: count, dtype: int64

In [7]:
# 1 = yes, 3 = no
# dropped: 2 = only during pregnancy, 4 = prediabetes, 7 = don't know, 9 = refused
diabetes_data = diabetes_data[diabetes_data["DIABETE4"].isin([1, 3])]

print("Rows before:", rows_before)
print("Rows after:", len(diabetes_data))
print("Rows removed:", rows_before - len(diabetes_data))

Rows before: 433323
Rows after: 418492
Rows removed: 14831


## Recode the features

In [8]:
# most BRFSS questions use 1 = yes and 2 = no, anything else (7, 9) becomes missing
def yes_no(column):
    return column.map({1: 1, 2: 0})


# 1 to 30 days, 88 = none, 77 = don't know, 99 = refused
def health_days(column):
    days = column.replace(88, 0)
    return days.where(days <= 30)

In [9]:
clean_data = pd.DataFrame()
clean_data["diabetes"] = (diabetes_data["DIABETE4"] == 1).astype(int)

# these two are calculated variables and use 1 = no, 2 = yes
clean_data["high_bp"] = diabetes_data["_RFHYPE6"].map({1: 0, 2: 1})
clean_data["high_chol"] = diabetes_data["_RFCHOL3"].map({1: 0, 2: 1})

# 1 = checked in the last 5 years, 2 = not in the last 5 years, 3 = never checked
clean_data["chol_check"] = diabetes_data["_CHOLCH3"].map({1: 1, 2: 0, 3: 0})

# _BMI5 is stored as BMI x 100
clean_data["bmi"] = diabetes_data["_BMI5"] / 100

clean_data["smoker"] = yes_no(diabetes_data["SMOKE100"])
clean_data["stroke"] = yes_no(diabetes_data["CVDSTRK3"])
clean_data["heart_disease"] = yes_no(diabetes_data["_MICHD"])
clean_data["phys_activity"] = yes_no(diabetes_data["EXERANY2"])
clean_data["cost_barrier"] = yes_no(diabetes_data["MEDCOST1"])
clean_data["diff_walk"] = yes_no(diabetes_data["DIFFWALK"])

In [10]:
# 1 to 10 are types of insurance, 88 = no insurance, 77 and 99 = don't know / refused
insurance_codes = {}
for code in range(1, 11):
    insurance_codes[code] = 1
insurance_codes[88] = 0

clean_data["has_insurance"] = diabetes_data["PRIMINS1"].map(insurance_codes)

In [11]:
# 1 = excellent ... 5 = poor
clean_data["gen_health"] = diabetes_data["GENHLTH"].where(diabetes_data["GENHLTH"] <= 5)
clean_data["mental_health_days"] = health_days(diabetes_data["MENTHLTH"])
clean_data["physical_health_days"] = health_days(diabetes_data["PHYSHLTH"])

# 1 = male, 0 = female
clean_data["sex"] = diabetes_data["_SEX"].map({1: 1, 2: 0})

# 1 = 18-24 ... 13 = 80 or older, 14 = don't know
clean_data["age_group"] = diabetes_data["_AGEG5YR"].where(diabetes_data["_AGEG5YR"] <= 13)

# 9 = don't know / missing for both
clean_data["education"] = diabetes_data["_EDUCAG"].where(diabetes_data["_EDUCAG"] <= 4)
clean_data["income"] = diabetes_data["_INCOMG1"].where(diabetes_data["_INCOMG1"] <= 7)

## Check the result

In [12]:
print(clean_data.shape)
clean_data.head()

(418492, 19)


,diabetes,high_bp,high_chol,chol_check,bmi,smoker,stroke,heart_disease,phys_activity,cost_barrier,diff_walk,has_insurance,gen_health,mental_health_days,physical_health_days,sex,age_group,education,income
0,1,1.0,0.0,1.0,30.47,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,0.0,0.0,0,13.0,3.0,NaN
1,0,1.0,1.0,1.0,28.56,0.0,0.0,0.0,1.0,0.0,0.0,1.0,2.0,0.0,0.0,0,13.0,3.0,NaN
2,0,1.0,1.0,1.0,22.31,1.0,0.0,0.0,1.0,1.0,1.0,1.0,4.0,2.0,6.0,0,13.0,2.0,1.0
3,0,0.0,0.0,1.0,27.44,0.0,0.0,0.0,1.0,0.0,1.0,1.0,2.0,0.0,2.0,0,12.0,3.0,NaN
4,1,1.0,0.0,1.0,25.85,0.0,0.0,0.0,1.0,0.0,1.0,1.0,4.0,0.0,0.0,0,12.0,3.0,5.0


In [13]:
clean_data.describe().T

,count,mean,std,min,25%,50%,75%,max
diabetes,418492.0,0.142861,0.349931,0.00,0.00,0.00,0.00,1.00
high_bp,416846.0,0.405054,0.490903,0.00,0.00,0.00,1.00,1.00
high_chol,365076.0,0.415954,0.492886,0.00,0.00,0.00,1.00,1.00
chol_check,392928.0,0.904649,0.293699,0.00,1.00,1.00,1.00,1.00
bmi,379582.0,28.403704,6.498116,12.02,24.03,27.37,31.57,99.84
smoker,396716.0,0.385548,0.486725,0.00,0.00,0.00,1.00,1.00
stroke,417340.0,0.042071,0.200752,0.00,0.00,0.00,0.00,1.00
heart_disease,414415.0,0.084012,0.277407,0.00,0.00,0.00,0.00,1.00
phys_activity,417333.0,0.754496,0.430386,0.00,1.00,1.00,1.00,1.00
cost_barrier,417093.0,0.084665,0.278382,0.00,0.00,0.00,0.00,1.00


In [14]:
missing = pd.DataFrame({
    "missing": clean_data.isna().sum(),
    "percent": (clean_data.isna().mean() * 100).round(2),
})
missing.sort_values("missing", ascending=False)

,missing,percent
income,83361,19.92
high_chol,53416,12.76
bmi,38910,9.30
chol_check,25564,6.11
smoker,21776,5.20
has_insurance,17953,4.29
diff_walk,17150,4.10
physical_health_days,10152,2.43
mental_health_days,7622,1.82
age_group,7401,1.77


In [15]:
print("Rows with at least one missing value:", clean_data.isna().any(axis=1).sum())

Rows with at least one missing value: 160016


In [16]:
clean_data["diabetes"].value_counts()

diabetes
0    358706
1     59786
Name: count, dtype: int64

In [17]:
clean_data["diabetes"].value_counts(normalize=True).round(4)

diabetes
0    0.8571
1    0.1429
Name: proportion, dtype: float64

In [18]:
# kept on purpose: with only 20 coded answers, different people can give the same answers
print("Duplicate rows:", clean_data.duplicated().sum())

Duplicate rows: 9871


## Save

In [19]:
clean_data.to_csv(processed_data_folder + "/diabetes_clean.csv", index=False)
print("Saved", clean_data.shape)

Saved (418492, 19)
